# Scraping Anime-Sama - Catalogue et planning

## Objectif du notebook

Ce notebook récupère les informations publiques de catalogue disponibles sur `https://anime-sama.to/` :

- les œuvres du catalogue ;
- les titres, genres, types, langues et synopsis ;
- l'URL de chaque fiche ;
- quelques détails supplémentaires depuis les fiches ;
- le planning des sorties.


# Étape 1 - Installer les bibliothèques

Décommenter la ligne suivante si les bibliothèques ne sont pas encore installées.

In [30]:
# %pip install requests beautifulsoup4 pandas
%pip install playwright

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Étape 2 - Imports et paramètres

## Paramètres utilisateur

Modifier `MAX_CATALOGUE_ITEMS` et `MAX_FICHES_A_ENRICHIR` selon le volume voulu.

In [ ]:
import json
import re
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://anime-sama.to"
CATALOGUE_URL = urljoin(BASE_URL, "/catalogue/")
PLANNING_URL = urljoin(BASE_URL, "/planning/")

MAX_CATALOGUE_ITEMS = 80
MAX_FICHES_A_ENRICHIR = 20
MAX_SAISONS_PAR_FICHE = 20
PAUSE_SECONDS = 0.5

DOSSIER_SORTIE = Path("exports")
DOSSIER_SORTIE.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/137.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8",
}

# Étape 3 - Fonctions générales

## Requêtes HTTP et nettoyage du texte

Ces fonctions sont des fonctions utilitaires elles récupèrent des pages web, nettoient du texte, découpent des listes et analysent des liens.

In [32]:
def maintenant_iso():
    return datetime.now().isoformat(timespec="seconds")


def nettoyer_texte(texte):
    return re.sub(r"\s+", " ", texte or "").strip()


def fetch_soup(url, timeout=20):
    response = requests.get(url, headers=HEADERS, timeout=timeout)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def split_liste(valeur):
    valeur = nettoyer_texte(valeur)
    if not valeur:
        return []
    morceaux = re.split(r"\s*,\s*|\s+-\s+", valeur)
    return [m.strip() for m in morceaux if m.strip()]


def extraire_entre(texte, debut, fins):
    texte = nettoyer_texte(texte)
    index_debut = texte.find(debut)
    if index_debut == -1:
        return ""
    index_debut += len(debut)

    index_fin = len(texte)
    for fin in fins:
        position = texte.find(fin, index_debut)
        if position != -1:
            index_fin = min(index_fin, position)
    return nettoyer_texte(texte[index_debut:index_fin])


def premiere_image(soup):
    image = soup.select_one("img[src]")
    if not image:
        return None
    return urljoin(BASE_URL, image.get("src"))


def normaliser_url_catalogue(href):
    if not href:
        return None
    url = urljoin(BASE_URL, href)
    return url.split("#", 1)[0]


def est_lien_catalogue_oeuvre(href):
    url = normaliser_url_catalogue(href)
    if not url:
        return False
    if not url.startswith(BASE_URL + "/catalogue/"):
        return False
    return url.rstrip("/") != (BASE_URL + "/catalogue").rstrip("/")


# Étape 4 - Analyser la structure du catalogue

## Données récupérables sur la page catalogue

Chaque résultat du catalogue contient généralement :

- le titre ;
- les titres alternatifs dans le même bloc texte ;
- les genres ;
- les types ;
- les langues ;
- le synopsis ;
- l'URL de la fiche.

In [33]:
soup_catalogue = fetch_soup(CATALOGUE_URL)
print("Titre de page :", nettoyer_texte(soup_catalogue.title.get_text()) if soup_catalogue.title else "Sans titre")

liens_catalogue = [
    a for a in soup_catalogue.select("a[href]")
    if est_lien_catalogue_oeuvre(a.get("href", ""))
    and nettoyer_texte(a.get_text(" ", strip=True))
]

print("Liens de fiches trouvés :", len(liens_catalogue))
print("Exemple de bloc texte :")
print(nettoyer_texte(liens_catalogue[0].get_text(" ", strip=True))[:600] if liens_catalogue else "Aucun lien trouvé")

Titre de page : Catalogue | Anime-Sama - Streaming et catalogage d'animes et scans.
Liens de fiches trouvés : 48
Exemple de bloc texte :
#DRCL midnight children Genres Drame, Fantastique, Surnaturel Types Scans Langues VF Synopsis Dans un pays lointain de l'Est, il existe une créature qu'on ne remarque pas mais qui est pourtant bien là... Le Comte Dracula !


# Étape 5 - Scraper le catalogue

## Extraction des œuvres

On parcourt les cartes du catalogue et on normalise les informations dans un dictionnaire commun.

In [34]:
def parser_bloc_catalogue(a, rank):
    href = a.get("href")
    url = normaliser_url_catalogue(href)
    texte_lignes = [nettoyer_texte(ligne) for ligne in a.get_text("\n", strip=True).split("\n")]
    texte_lignes = [ligne for ligne in texte_lignes if ligne]
    texte_complet = nettoyer_texte(a.get_text(" ", strip=True))

    titre = texte_lignes[0] if texte_lignes else "Titre non trouvé"
    bloc_titres = extraire_entre(texte_complet, titre, ["Genres", "Types", "Langues", "Synopsis"])
    titres_alternatifs = split_liste(bloc_titres)

    genres = split_liste(extraire_entre(texte_complet, "Genres", ["Types", "Langues", "Synopsis"]))
    types = split_liste(extraire_entre(texte_complet, "Types", ["Langues", "Synopsis"]))
    langues = split_liste(extraire_entre(texte_complet, "Langues", ["Synopsis"]))
    synopsis = extraire_entre(texte_complet, "Synopsis", [])

    return {
        "rank": rank,
        "title": titre,
        "alternative_titles": titres_alternatifs,
        "genres": genres,
        "types": types,
        "languages": langues,
        "synopsis": synopsis,
        "url": url,
        "source": "Anime-Sama catalogue",
        "collected_at": maintenant_iso(),
    }


def scraper_catalogue(max_items=80):
    soup = fetch_soup(CATALOGUE_URL)
    liens = []
    urls_vues = set()

    for a in soup.select("a[href]"):
        href = a.get("href", "")
        if not est_lien_catalogue_oeuvre(href):
            continue
        url = normaliser_url_catalogue(href)
        texte = nettoyer_texte(a.get_text(" ", strip=True))
        if not texte or url in urls_vues:
            continue
        urls_vues.add(url)
        liens.append(a)

    resultats = []
    for rank, a in enumerate(liens[:max_items], start=1):
        resultats.append(parser_bloc_catalogue(a, rank))

    return resultats


catalogue = scraper_catalogue(MAX_CATALOGUE_ITEMS)
df_catalogue = pd.DataFrame(catalogue)
print("Œuvres récupérées :", len(df_catalogue))
if df_catalogue.empty:
    print("Aucun résultat : vérifiez que l'étape 4 affiche bien des liens de fiches.")
df_catalogue.head(10)

Œuvres récupérées : 48


,rank,title,alternative_titles,genres,types,languages,synopsis,url,source,collected_at
0,1,#DRCL midnight children,[],"[Drame, Fantastique, Surnaturel]",[Scans],[VF],"Dans un pays lointain de l'Est, il existe une ...",https://anime-sama.to/catalogue/drcl-midnight-...,Anime-Sama catalogue,2026-06-05T11:46:19
1,2,"'Tis Time for ""Torture,"" Princess","['Tis Time for ""Torture, "" Princess, Tis Time ...","[Comédie, Fantasy, Démons, Gastronomie, Tortur...",[Anime],[VOSTFR],"Hime-sama, princesse de l'armée royale, et son...",https://anime-sama.to/catalogue/tis-time-for-t...,Anime-Sama catalogue,2026-06-05T11:46:19
2,3,100 Jours Avant Ta Mort,"[Last Hundred Days Till She Dies, Kimi ga Shin...","[Comédie, Drame, Romance, School Life, Slice o...",[Scans],[VF],Taro a enfin pu avouer à son amie d'enfance Um...,https://anime-sama.to/catalogue/100-jours-avan...,Anime-Sama catalogue,2026-06-05T11:46:19
3,4,100 Meters,"[100 mètres, Hyakuemu]","[Action, Drame, Athlétisme, Sport, Drame]",[Film],"[VOSTFR, VF, VASTFR]",Je m'appelle Togashi. Je suis né avec un don p...,https://anime-sama.to/catalogue/100-meters,Anime-Sama catalogue,2026-06-05T11:46:19
4,5,16bit Sensation: Another Layer,[16bit Sensation: Watashi to Minna ga Tsukutta...,"[Comédie, Slice of Life, Art, Jeux vidéo, Nost...",[Anime],[VOSTFR],"En 1992, une étudiante universitaire, Meiko Ue...",https://anime-sama.to/catalogue/16bit-sensatio...,Anime-Sama catalogue,2026-06-05T11:46:19
5,6,2.43 Seiin Koukou Danshi Volley-bu,"[2.43: Seiin High School Boys Volleyball Team,...","[Comedie, Josei, School Life, Tournois, Drame]",[Anime],[VOSTFR],"Kimichika Haijima, étudiant au secondaire, ret...",https://anime-sama.to/catalogue/2.43-seiin-kou...,Anime-Sama catalogue,2026-06-05T11:46:19
6,7,2.5-jigen no Ririsa,"[2.5 Dimensional Seduction, 2.5 Jigen no Yuwak...","[Comédie, Ecchi, Mature, Romance, School Life,...",[Anime],[VOSTFR],"""Je ne suis pas intéressé par les vraies femme...",https://anime-sama.to/catalogue/2.5-jigen-no-r...,Anime-Sama catalogue,2026-06-05T11:46:19
7,8,20th Century Boys,[21st Century Boys],"[Action, Drame, Mystère, Surnaturel, Thriller,...",[Scans],[VF],1969 : Un groupe d'enfants se retrouve réguliè...,https://anime-sama.to/catalogue/20th-century-boys,Anime-Sama catalogue,2026-06-05T11:46:19
8,9,29-Sai Dokushin Chuuken Boukensha no Nichijou,"[An Adventurer's Daily Grind at Age 29, 29-sai...","[Aventure, Fantasy, Slice of Life, Magie, Scie...",[Anime],[VOSTFR],Hajime Shinonome est un aventurier qui fait pa...,https://anime-sama.to/catalogue/29-sai-dokushi...,Anime-Sama catalogue,2026-06-05T11:46:19
9,10,365 Days To The Wedding,"[Kekkon Surutte, Hontou desu ka?, Kekkon Surut...","[Romance, Slice of Life, Mariage, Travail, Com...",[Anime],[VOSTFR],Takuya et Rika travaillent ensemble dans une a...,https://anime-sama.to/catalogue/365-days-to-th...,Anime-Sama catalogue,2026-06-05T11:46:19


# Étape 6 - Scraper les fiches œuvres

## Enrichissement depuis chaque fiche

Sur une fiche, on peut récupérer plus proprement :

- le titre principal ;
- les titres alternatifs ;
- l'image ;
- le synopsis ;
- les genres ;
- les œuvres similaires visibles.

In [35]:
import re
import time
import requests
import pandas as pd
from urllib.parse import urljoin


def extraire_section_apres_titre(soup, titre_section):
    pattern = re.compile(rf"^{re.escape(titre_section)}$", re.I)
    titre = soup.find(lambda tag: tag.name in ["h1", "h2", "h3", "h4"] and pattern.search(nettoyer_texte(tag.get_text())))
    if not titre:
        return ""
    morceaux = []
    for sibling in titre.find_all_next():
        if sibling.name in ["h1", "h2", "h3", "h4"]:
            break
        texte = nettoyer_texte(sibling.get_text(" ", strip=True))
        if texte:
            morceaux.append(texte)
    return nettoyer_texte(" ".join(morceaux))


def extraire_saisons_depuis_fiche(soup, fiche_url):
    """Récupère les saisons depuis les appels panneauAnime('Saison 1', 'saison1/vostfr')."""
    saisons = []
    urls_vues = set()
    for script in soup.select("script"):
        contenu = script.get_text("\n", strip=False)
        for label, href in re.findall(r"panneauAnime\(\s*['\"]([^'\"]+)['\"]\s*,\s*['\"]([^'\"]+)['\"]\s*\)", contenu):
            label = nettoyer_texte(label)
            href = nettoyer_texte(href)
            if label.lower() == "nom" and href.lower() == "url":
                continue
            season_url = urljoin(fiche_url if fiche_url.endswith("/") else fiche_url + "/", href)
            if season_url in urls_vues:
                continue
            urls_vues.add(season_url)
            saisons.append({"season_label": nettoyer_texte(label), "season_url": season_url})
    return saisons


def extraire_iframes_depuis_js(js_text):
    """Extrait les URLs de lecteurs depuis episodes.js -> liste d'épisodes."""
    tableaux = []
    for nom, bloc in re.findall(r"var\s+(eps\w+)\s*=\s*\[(.*?)\]\s*;", js_text, flags=re.S):
        urls = [u.strip() for u in re.findall(r"['\"]([^'\"]+)['\"]", bloc) if u.strip()]
        if urls:
            tableaux.append((nom, urls))
    if not tableaux:
        return []
    nb_episodes = max(len(urls) for _, urls in tableaux)
    episodes = []
    for i in range(nb_episodes):
        lecteurs = [urls[i] for _, urls in tableaux if i < len(urls)]
        episodes.append({
            "episode": i + 1,
            "iframe": lecteurs[0] if lecteurs else None,
            "lecteurs": lecteurs,
        })
    return episodes


def recuperer_iframes_saison(season_url):
    """Télécharge episodes.js d'une saison et renvoie les épisodes avec leurs iframes."""
    soup = fetch_soup(season_url)
    script = soup.select_one("script[src*='episodes.js']")
    if not script:
        return []
    js_url = urljoin(season_url if season_url.endswith("/") else season_url + "/", script.get("src"))
    response = requests.get(js_url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    return extraire_iframes_depuis_js(response.text)


def compter_episodes_saison(season_url):
    soup = fetch_soup(season_url)
    options = soup.select("#selectEpisodes option")
    if options:
        return len(options)
    script = soup.select_one("script[src*='episodes.js']")
    if not script:
        return 0
    js_url = urljoin(season_url if season_url.endswith("/") else season_url + "/", script.get("src"))
    response = requests.get(js_url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    longueurs = []
    for bloc in re.findall(r"var\s+eps\w+\s*=\s*\[(.*?)\]\s*;", response.text, flags=re.S):
        nombre = len(re.findall(r"['\"][^'\"]+['\"]", bloc))
        if nombre:
            longueurs.append(nombre)
    return max(longueurs) if longueurs else 0


def enrichir_saisons_avec_episodes(saisons, max_saisons=21):
    saisons_enrichies = []
    total = 0
    for saison in saisons[:max_saisons]:
        saison = dict(saison)
        try:
            episode_count = compter_episodes_saison(saison["season_url"])
        except Exception as erreur:
            episode_count = 0
            saison["episode_error"] = str(erreur)
        saison["episode_count"] = episode_count
        total += episode_count
        saisons_enrichies.append(saison)
        time.sleep(PAUSE_SECONDS)
    return saisons_enrichies, total


def scraper_fiche_oeuvre(url):
    soup = fetch_soup(url)
    texte_page = nettoyer_texte(soup.get_text(" ", strip=True))

    titre_tag = soup.find(["h1", "h2", "h3", "h4"])
    titre = nettoyer_texte(titre_tag.get_text(" ", strip=True)) if titre_tag else None

    synopsis = extraire_section_apres_titre(soup, "Synopsis")
    genres_texte = extraire_section_apres_titre(soup, "Genres")
    genres = split_liste(genres_texte)

    saisons = extraire_saisons_depuis_fiche(soup, url)
    saisons_enrichies, total_episodes = enrichir_saisons_avec_episodes(saisons, MAX_SAISONS_PAR_FICHE)

    for saison in saisons_enrichies:
        try:
            saison["iframes"] = recuperer_iframes_saison(saison["season_url"])
        except Exception as e:
            saison["iframes"] = []
            saison["iframe_error"] = repr(e)
        time.sleep(PAUSE_SECONDS)

    similaires = []
    for a in soup.select("a[href]"):
        if not est_lien_catalogue_oeuvre(a.get("href", "")):
            continue
        lien = normaliser_url_catalogue(a.get("href"))
        texte = nettoyer_texte(a.get_text(" ", strip=True))
        if lien != url and texte:
            similaires.append({"title": texte.split(" Genres ")[0], "url": lien})

    return {
        "url": url,
        "fiche_title": titre,
        "fiche_image_url": premiere_image(soup),
        "fiche_synopsis": synopsis,
        "fiche_genres": genres,
        "season_count": len(saisons),
        "total_episode_count": total_episodes,
        "seasons": saisons,
        "episodes_by_season": saisons_enrichies,
        "similar_titles": similaires[:10],
        "page_text_sample": texte_page[:300],
        "collected_at_fiche": maintenant_iso(),
    }


fiches = []
for index, row in df_catalogue.head(MAX_FICHES_A_ENRICHIR).iterrows():
    try:
        fiche = scraper_fiche_oeuvre(row["url"])
        fiches.append(fiche)
        print(f"[{len(fiches)}/{MAX_FICHES_A_ENRICHIR}] {row['title']}")
        time.sleep(PAUSE_SECONDS)
    except Exception as erreur:
        print(f"Fiche ignorée : {row['url']} -> {type(erreur).__name__}: {erreur!r}")


df_fiches = pd.DataFrame(fiches)
df_fiches.head(10)

[1/48] #DRCL midnight children
[2/48] 'Tis Time for "Torture," Princess
[3/48] 100 Jours Avant Ta Mort
[4/48] 100 Meters
[5/48] 16bit Sensation: Another Layer
[6/48] 2.43 Seiin Koukou Danshi Volley-bu
[7/48] 2.5-jigen no Ririsa
[8/48] 20th Century Boys
[9/48] 29-Sai Dokushin Chuuken Boukensha no Nichijou
[10/48] 365 Days To The Wedding
[11/48] 3D Kanojo
[12/48] 4 Cut Hero
[13/48] 5cm per second
[14/48] 66,666 Years Advent of the Dark Mage
[15/48] 7 Jours
[16/48] 7 Seeds
[17/48] 7th Time Loop: The Villainess Enjoys a Carefree Life Married to Her Worst Enemy!
[18/48] 86 Eighty Six
[19/48] 91 Days
[20/48] 99 Reinforced Wood Stick
[21/48] A Barbarian Was Admitted to the Academy
[22/48] A Barbarian's Adventure in a Fantasy World
[23/48] A Blade At A Time
[24/48] A Centaur's Life
[25/48] A Certain Magical Index
[26/48] A Certain Scientific Accelerator
[27/48] A Certain Scientific Railgun
[28/48] A Condition Called Love
[29/48] A Couple of Cuckoos
[30/48] A Destructive God Sits Next to Me
[31

,url,fiche_title,fiche_image_url,fiche_synopsis,fiche_genres,season_count,total_episode_count,seasons,episodes_by_season,similar_titles,page_text_sample,collected_at_fiche
0,https://anime-sama.to/catalogue/drcl-midnight-...,Aperçu,https://raw.githubusercontent.com/Anime-Sama/I...,"Dans un pays lointain de l'Est, il existe une ...",[Drame Fantastique Surnaturel Drame Fantastiqu...,0,0,[],[],"[{'title': 'Black Torch', 'url': 'https://anim...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:20
1,https://anime-sama.to/catalogue/tis-time-for-t...,Aperçu,https://raw.githubusercontent.com/Anime-Sama/I...,"Hime-sama, princesse de l'armée royale, et son...",[Comédie Fantasy Démons Gastronomie Torture Sc...,2,24,"[{'season_label': 'Saison 1', 'season_url': 'h...","[{'season_label': 'Saison 1', 'season_url': 'h...","[{'title': 'Pass the Monster Meat, Milady! Aku...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:27
2,https://anime-sama.to/catalogue/100-jours-avan...,Aperçu,https://raw.githubusercontent.com/Anime-Sama/I...,Taro a enfin pu avouer à son amie d'enfance Um...,[Comédie Drame Romance School Life Slice of Li...,0,0,[],[],"[{'title': '3D Kanojo 3D Kanojo: Real Girl, 3D...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:27
3,https://anime-sama.to/catalogue/100-meters,Aperçu,https://raw.githubusercontent.com/Anime-Sama/I...,Je m'appelle Togashi. Je suis né avec un don p...,[Action Drame Athlétisme Sport Drame Action Dr...,1,1,"[{'season_label': 'Film', 'season_url': 'https...","[{'season_label': 'Film', 'season_url': 'https...","[{'title': 'Ascension Kokou no Hito, The Climb...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:31
4,https://anime-sama.to/catalogue/16bit-sensatio...,Aperçu,https://raw.githubusercontent.com/Anime-Sama/I...,"En 1992, une étudiante universitaire, Meiko Ue...",[Comédie Slice of Life Art Jeux vidéo Nostalgi...,1,13,"[{'season_label': 'Saison 1', 'season_url': 'h...","[{'season_label': 'Saison 1', 'season_url': 'h...","[{'title': 'Dragon Ball GT DBGT, Dragonball Gr...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:35
5,https://anime-sama.to/catalogue/2.43-seiin-kou...,Aperçu,https://raw.githubusercontent.com/Anime-Sama/I...,"Kimichika Haijima, étudiant au secondaire, ret...",[Comedie Josei School Life Tournois Drame Come...,1,12,"[{'season_label': 'Saison 1', 'season_url': 'h...","[{'season_label': 'Saison 1', 'season_url': 'h...",[{'title': 'Gunjou no Fanfare Fanfare of Adole...,Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:39
6,https://anime-sama.to/catalogue/2.5-jigen-no-r...,Aperçu,https://raw.githubusercontent.com/Anime-Sama/I...,"""Je ne suis pas intéressé par les vraies femme...",[Comédie Ecchi Mature Romance School Life Shôn...,1,24,"[{'season_label': 'Saison 1', 'season_url': 'h...","[{'season_label': 'Saison 1', 'season_url': 'h...",[{'title': 'Boku wa Tomodachi ga Sukunai Hagan...,Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:43
7,https://anime-sama.to/catalogue/20th-century-boys,Aperçu,https://raw.githubusercontent.com/Anime-Sama/I...,1969 : Un groupe d'enfants se retrouve réguliè...,[Action Drame Mystère Surnaturel Thriller Sein...,0,0,[],[],"[{'title': 'Adabana', 'url': 'https://anime-sa...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:44
8,https://anime-sama.to/catalogue/29-sai-dokushi...,Aperçu,https://raw.githubusercontent.com/Anime-Sama/I...,Hajime Shinonome est un aventurier qui fait pa...,[Aventure Fantasy Slice of Life Magie Science ...,12,15,"[{'season_label': 'Saison 1', 'season_url': 'h...","[{'season_label': 'Saison 1', 'season_url': 'h...",[{'title': 'An Archdemon's Dilemma: How to Lov...,Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:47:15
9,https://anime-sama.to/catalogue/365-days-to-th...,Aperçu,https://raw.githubusercontent.com/Anime-Sama/I...,Takuya et Rika travaillent ensemble dans une a...,[Romance

# Étape 7 - Fusionner catalogue et fiches

## Résultat enrichi

On fusionne les informations rapides du catalogue avec les informations détaillées récupérées dans les fiches.

In [36]:
SCHEMA_OEUVRES = [
    "rank", "title", "alternative_titles", "genres", "types", "languages", "synopsis",
    "url", "source", "collected_at", "fiche_title", "fiche_image_url",
    "fiche_synopsis", "fiche_genres", "season_count", "total_episode_count",
    "seasons", "episodes_by_season", "similar_titles", "page_text_sample",
    "collected_at_fiche"
]

if not df_catalogue.empty and not df_fiches.empty:
    df_oeuvres = df_catalogue.merge(df_fiches, on="url", how="left")
else:
    df_oeuvres = df_catalogue.copy()

# Garantit que les colonnes existent même si le scraping n'a rien récupéré.
for colonne in SCHEMA_OEUVRES:
    if colonne not in df_oeuvres.columns:
        df_oeuvres[colonne] = pd.NA

df_oeuvres = df_oeuvres[SCHEMA_OEUVRES]

if df_oeuvres.empty:
    print("Aucune œuvre récupérée pour le moment. Relancez d'abord l'étape 5 - Scraper le catalogue.")
else:
    print("Œuvres disponibles dans df_oeuvres :", len(df_oeuvres))

df_oeuvres.head(10)

Œuvres disponibles dans df_oeuvres : 48


,rank,title,alternative_titles,genres,types,languages,synopsis,url,source,collected_at,...,fiche_image_url,fiche_synopsis,fiche_genres,season_count,total_episode_count,seasons,episodes_by_season,similar_titles,page_text_sample,collected_at_fiche
0,1,#DRCL midnight children,[],"[Drame, Fantastique, Surnaturel]",[Scans],[VF],"Dans un pays lointain de l'Est, il existe une ...",https://anime-sama.to/catalogue/drcl-midnight-...,Anime-Sama catalogue,2026-06-05T11:46:19,...,https://raw.githubusercontent.com/Anime-Sama/I...,"Dans un pays lointain de l'Est, il existe une ...",[Drame Fantastique Surnaturel Drame Fantastiqu...,0,0,[],[],"[{'title': 'Black Torch', 'url': 'https://anim...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:20
1,2,"'Tis Time for ""Torture,"" Princess","['Tis Time for ""Torture, "" Princess, Tis Time ...","[Comédie, Fantasy, Démons, Gastronomie, Tortur...",[Anime],[VOSTFR],"Hime-sama, princesse de l'armée royale, et son...",https://anime-sama.to/catalogue/tis-time-for-t...,Anime-Sama catalogue,2026-06-05T11:46:19,...,https://raw.githubusercontent.com/Anime-Sama/I...,"Hime-sama, princesse de l'armée royale, et son...",[Comédie Fantasy Démons Gastronomie Torture Sc...,2,24,"[{'season_label': 'Saison 1', 'season_url': 'h...","[{'season_label': 'Saison 1', 'season_url': 'h...","[{'title': 'Pass the Monster Meat, Milady! Aku...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:27
2,3,100 Jours Avant Ta Mort,"[Last Hundred Days Till She Dies, Kimi ga Shin...","[Comédie, Drame, Romance, School Life, Slice o...",[Scans],[VF],Taro a enfin pu avouer à son amie d'enfance Um...,https://anime-sama.to/catalogue/100-jours-avan...,Anime-Sama catalogue,2026-06-05T11:46:19,...,https://raw.githubusercontent.com/Anime-Sama/I...,Taro a enfin pu avouer à son amie d'enfance Um...,[Comédie Drame Romance School Life Slice of Li...,0,0,[],[],"[{'title': '3D Kanojo 3D Kanojo: Real Girl, 3D...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:27
3,4,100 Meters,"[100 mètres, Hyakuemu]","[Action, Drame, Athlétisme, Sport, Drame]",[Film],"[VOSTFR, VF, VASTFR]",Je m'appelle Togashi. Je suis né avec un don p...,https://anime-sama.to/catalogue/100-meters,Anime-Sama catalogue,2026-06-05T11:46:19,...,https://raw.githubusercontent.com/Anime-Sama/I...,Je m'appelle Togashi. Je suis né avec un don p...,[Action Drame Athlétisme Sport Drame Action Dr...,1,1,"[{'season_label': 'Film', 'season_url': 'https...","[{'season_label': 'Film', 'season_url': 'https...","[{'title': 'Ascension Kokou no Hito, The Climb...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:31
4,5,16bit Sensation: Another Layer,[16bit Sensation: Watashi to Minna ga Tsukutta...,"[Comédie, Slice of Life, Art, Jeux vidéo, Nost...",[Anime],[VOSTFR],"En 1992, une étudiante universitaire, Meiko Ue...",https://anime-sama.to/catalogue/16bit-sensatio...,Anime-Sama catalogue,2026-06-05T11:46:19,...,https://raw.githubusercontent.com/Anime-Sama/I...,"En 1992, une étudiante universitaire, Meiko Ue...",[Comédie Slice of Life Art Jeux vidéo Nostalgi...,1,13,"[{'season_label': 'Saison 1', 'season_url': 'h...","[{'season_label': 'Saison 1', 'season_url': 'h...","[{'title': 'Dragon Ball GT DBGT, Dragonball Gr...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:35
5,6,2.43 Seiin Koukou Danshi Volley-bu,"[2.43: Seiin High School Boys Volleyball Team,...","[Comedie, Josei, School Life, Tournois, Drame]",[Anime],[VOSTFR],"Kimichika Haijima, étudiant au secondaire, ret...",https://anime-sama.to/catalogue/2.43-seiin-kou...,Anime-Sama catalogue,2026-06-05T11:46:19,...,https://raw.githubusercontent.com/Anime-Sama/I...,"Kimichika Haijima, étudiant au secondaire, ret...",[Comedie Josei School Life Tournois Drame Come...,1,12,"[{'season_label': 'Saison 1', 'season_url': 'h...","[{'season_label': 'Saison 1', 'season_url': 'h...",[{'title': 'Gunjou no Fanfare Fanfare of Adole...,Catalogue Planning Aide Profil (Déconnecté) Ca

# Étape 8 - Scraper le planning

## Données récupérées

Le planning contient les sorties par jour :

- jour ;
- date ;
- type ;
- titre ;
- heure ;
- saison ou information complémentaire ;
- URL de la fiche.

In [37]:
def parser_entree_planning(texte):
    texte = nettoyer_texte(texte)
    type_oeuvre = None
    if texte.startswith("Anime "):
        type_oeuvre = "Anime"
        texte = texte[len("Anime "):]
    elif texte.startswith("Scans "):
        type_oeuvre = "Scans"
        texte = texte[len("Scans "):]

    match_heure = re.search(r"\b(\d{1,2}h\d{2}|\?)\b", texte)
    heure = match_heure.group(1) if match_heure else None

    titre = texte
    detail = None
    if match_heure:
        titre = nettoyer_texte(texte[:match_heure.start()])
        detail = nettoyer_texte(texte[match_heure.end():])

    return type_oeuvre, titre, heure, detail


def scraper_planning():
    soup = fetch_soup(PLANNING_URL)
    resultats = []
    jour_courant = None
    date_courante = None

    for element in soup.find_all(["h2", "h3", "a"]):
        texte = nettoyer_texte(element.get_text(" ", strip=True))
        if not texte:
            continue

        if element.name in ["h2", "h3"] and texte in ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]:
            jour_courant = texte
            date_courante = None
            continue

        if re.fullmatch(r"\d{2}/\d{2}", texte):
            date_courante = texte
            continue

        if element.name == "a" and est_lien_catalogue_oeuvre(element.get("href", "")):
            type_oeuvre, titre, heure, detail = parser_entree_planning(texte)
            resultats.append({
                "day": jour_courant,
                "date": date_courante,
                "type": type_oeuvre,
                "title": titre,
                "time": heure,
                "detail": detail,
                "url": normaliser_url_catalogue(element.get("href")),
                "source": "Anime-Sama planning",
                "collected_at": maintenant_iso(),
            })

    return resultats


planning = scraper_planning()
df_planning = pd.DataFrame(planning)
print("Sorties récupérées :", len(df_planning))
df_planning.head(20)

Sorties récupérées : 269


,day,date,type,title,time,detail,url,source,collected_at
0,Lundi,None,Anime,Rick et Morty,13h00,Saison 9,https://anime-sama.to/catalogue/rick-et-morty/...,Anime-Sama planning,2026-06-05T11:50:50
1,Lundi,None,Anime,Rick et Morty,13h00,Saison 9,https://anime-sama.to/catalogue/rick-et-morty/...,Anime-Sama planning,2026-06-05T11:50:50
2,Lundi,None,Anime,L'Atelier des Sorciers,16h20,Saison 1,https://anime-sama.to/catalogue/l-atelier-des-...,Anime-Sama planning,2026-06-05T11:50:50
3,Lundi,None,Anime,Farming Life in Another World,16h20,Saison 2,https://anime-sama.to/catalogue/farming-life-i...,Anime-Sama planning,2026-06-05T11:50:50
4,Lundi,None,Anime,L'Atelier des Sorciers,17h15,Saison 1,https://anime-sama.to/catalogue/l-atelier-des-...,Anime-Sama planning,2026-06-05T11:50:50
5,Lundi,None,Anime,Ponkotsu Fuukiin to Skirt Take ga Futekisetsu ...,17h20,Saison 1,https://anime-sama.to/catalogue/ponkotsu-fuuki...,Anime-Sama planning,2026-06-05T11:50:50
6,Lundi,None,Anime,Liar Game,18h20,Saison 1,https://anime-sama.to/catalogue/liar-game/sais...,Anime-Sama planning,2026-06-05T11:50:50
7,Lundi,None,Scans,Blue Box ?,None,None,https://anime-sama.to/catalogue/blue-box/scan/vf,Anime-Sama planning,2026-06-05T11:50:50
8,Lundi,None,Scans,Ordeal ?,None,None,https://anime-sama.to/catalogue/ordeal/scan/vf,Anime-Sama planning,2026-06-05T11:50:50
9,Lundi,None,Scans,Juvenile Detention Center ?,None,None,https://anime-sama.to/catalogue/juvenile-deten...,Anime-Sama planning,2026-06-05T11:50:50


# Étape 9 - Rechercher dans les données

## Exemple de recherche locale

Une fois les données scrapées, on peut filtrer le tableau sans relancer le scraping.

In [38]:
def colonne_recherche(df, colonne):
    if colonne not in df.columns:
        return pd.Series("", index=df.index, dtype="string")
    return df[colonne].fillna("").astype(str)


def rechercher_oeuvre(df, terme):
    if df.empty:
        print("df_oeuvres est vide : relancez les étapes de scraping avant la recherche.")
        return df.copy()

    terme = str(terme).lower()
    texte_recherche = (
        colonne_recherche(df, "title") + " "
        + colonne_recherche(df, "synopsis") + " "
        + colonne_recherche(df, "genres") + " "
        + colonne_recherche(df, "languages") + " "
        + colonne_recherche(df, "types")
    ).str.lower()

    masque = texte_recherche.str.contains(re.escape(terme), na=False)
    return df[masque].copy()


rechercher_oeuvre(df_oeuvres, "fantasy").head(10)

,rank,title,alternative_titles,genres,types,languages,synopsis,url,source,collected_at,...,fiche_image_url,fiche_synopsis,fiche_genres,season_count,total_episode_count,seasons,episodes_by_season,similar_titles,page_text_sample,collected_at_fiche
1,2,"'Tis Time for ""Torture,"" Princess","['Tis Time for ""Torture, "" Princess, Tis Time ...","[Comédie, Fantasy, Démons, Gastronomie, Tortur...",[Anime],[VOSTFR],"Hime-sama, princesse de l'armée royale, et son...",https://anime-sama.to/catalogue/tis-time-for-t...,Anime-Sama catalogue,2026-06-05T11:46:19,...,https://raw.githubusercontent.com/Anime-Sama/I...,"Hime-sama, princesse de l'armée royale, et son...",[Comédie Fantasy Démons Gastronomie Torture Sc...,2,24,"[{'season_label': 'Saison 1', 'season_url': 'h...","[{'season_label': 'Saison 1', 'season_url': 'h...","[{'title': 'Pass the Monster Meat, Milady! Aku...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:46:27
8,9,29-Sai Dokushin Chuuken Boukensha no Nichijou,"[An Adventurer's Daily Grind at Age 29, 29-sai...","[Aventure, Fantasy, Slice of Life, Magie, Scie...",[Anime],[VOSTFR],Hajime Shinonome est un aventurier qui fait pa...,https://anime-sama.to/catalogue/29-sai-dokushi...,Anime-Sama catalogue,2026-06-05T11:46:19,...,https://raw.githubusercontent.com/Anime-Sama/I...,Hajime Shinonome est un aventurier qui fait pa...,[Aventure Fantasy Slice of Life Magie Science ...,12,15,"[{'season_label': 'Saison 1', 'season_url': 'h...","[{'season_label': 'Saison 1', 'season_url': 'h...",[{'title': 'An Archdemon's Dilemma: How to Lov...,Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:47:15
11,12,4 Cut Hero,"[Si Ge Yongzhe, Sì Gé Yong Zhe, Si Ge Yong Zhe...","[Action, Aventure, Comédie, Fantasy, Combats, ...",[Anime],[VOSTFR],"Après avoir vaincu le Roi Démon, notre héros s...",https://anime-sama.to/catalogue/4-cut-hero,Anime-Sama catalogue,2026-06-05T11:46:19,...,https://raw.githubusercontent.com/Anime-Sama/I...,"Après avoir vaincu le Roi Démon, notre héros s...",[Action Aventure Comédie Fantasy Combats Démon...,1,10,"[{'season_label': 'Saison 1', 'season_url': 'h...","[{'season_label': 'Saison 1', 'season_url': 'h...","[{'title': 'New Saga Tsuyokute New Saga, Be St...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:47:49
13,14,"66,666 Years Advent of the Dark Mage","[66666nyeon Mane Hwansaenghan Heukmabeopsa, 66...","[Webcomic, Manhwa, Action, Fantasy, Combats, M...",[Scans],[],"Diablo Volpir, un puissant magicien, a été sce...",https://anime-sama.to/catalogue/66666-years-ad...,Anime-Sama catalogue,2026-06-05T11:46:19,...,https://raw.githubusercontent.com/Anime-Sama/I...,"Diablo Volpir, un puissant magicien, a été sce...",[Webcomic Manhwa Action Fantasy Combats Magie ...,0,0,[],[],"[{'title': 'Dungeon Reset', 'url': 'https://an...",Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:47:54
16,17,7th Time Loop: The Villainess Enjoys a Carefre...,"[7ème boucle temporelle, La méchante jouit d'u...","[Fantasy, Romance, Royauté, Voyage temporel, D...",[Anime],[VOSTFR],"Contrairement aux apparences, la duchesse Rish...",https://anime-sama.to/catalogue/7th-time-loop-...,Anime-Sama catalogue,2026-06-05T11:46:19,...,https://raw.githubusercontent.com/Anime-Sama/I...,"Contrairement aux apparences, la duchesse Rish...",[Fantasy Romance Royauté Voyage temporel Drame...,1,12,"[{'season_label': 'Saison 1', 'season_url': 'h...","[{'season_label': 'Saison 1', 'season_url': 'h...",[{'title': 'La Princesse et la Bête Niehime to...,Catalogue Planning Aide Profil (Déconnecté) Ca...,2026-06-05T11:48:06
21,22,A Barbarian's Adventure in a Fantasy World,[Surviving as a Barbarian in a Fantasy World],"[Manhwa, Webcomic, Shônen, Action, Aventure, C...",[Scans],[VF],"Chaque nuit, il priait pour se réveiller dans ...",https://anime-sama.to/catalogue/a-barbarians-a...,Anime-Sama catalogue,2026-06-05T11:46:19,...,https://raw.githubusercontent.com/Anime-Sama/I...,"Chaque nuit, il priait pour se réveiller dans ..

In [39]:
%pip install pymongo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from pymongo import MongoClient, UpdateOne


def stocker_dans_mongo(
    df_catalogue, df_oeuvres, df_planning,
    uri="mettre_uri_de_bdd_mongoDb_ici",
    db_name="anime_sama",
):
    """Stocke les DataFrames dans MongoDB (upsert pour éviter les doublons)."""
    client = MongoClient(uri)
    db = client[db_name]

    # On part des DataFrames NON aplatis -> listes/dicts conservés en natif
    collections = {
        "catalogue": (df_catalogue, "url"),
        "oeuvres": (df_oeuvres, "url"),
        "planning": (df_planning, None),
    }

    for nom, (df, cle) in collections.items():
        col = db[nom]
        records = df.to_dict(orient="records")
        if not records:
            print(f"  {nom}: aucun document")
            continue

        if cle and all(cle in r for r in records):
            # Upsert : met à jour si la clé existe déjà, insère sinon
            ops = [
                UpdateOne({cle: r[cle]}, {"$set": r}, upsert=True)
                for r in records
            ]
            resultat = col.bulk_write(ops, ordered=False)
            print(f"  {nom}: {resultat.upserted_count} insérés, "
                  f"{resultat.modified_count} mis à jour")
            col.create_index(cle, unique=True)
        else:
            # Pas de clé naturelle : on remplace tout le contenu
            col.delete_many({})
            col.insert_many(records)
            print(f"  {nom}: {len(records)} documents insérés (remplacement)")

    client.close()


print("\nStockage MongoDB :")
stocker_dans_mongo(df_catalogue, df_oeuvres, df_planning)


Stockage MongoDB :
  catalogue: 0 insérés, 48 mis à jour
  oeuvres: 0 insérés, 48 mis à jour
  planning: 269 documents insérés (remplacement)


# Étape 10 - Exporter les résultats

## Fichiers générés

Les données sont exportées en CSV et JSON dans le dossier `exports`.

In [41]:
fichier_catalogue_csv = DOSSIER_SORTIE / "anime_sama_catalogue.csv"
fichier_catalogue_json = DOSSIER_SORTIE / "anime_sama_catalogue.json"
fichier_oeuvres_csv = DOSSIER_SORTIE / "anime_sama_oeuvres_enrichies.csv"
fichier_oeuvres_json = DOSSIER_SORTIE / "anime_sama_oeuvres_enrichies.json"
fichier_planning_csv = DOSSIER_SORTIE / "anime_sama_planning.csv"
fichier_planning_json = DOSSIER_SORTIE / "anime_sama_planning.json"

# CSV : les listes sont converties en texte pour rester lisibles dans Excel.
df_catalogue_export = df_catalogue.copy()
df_oeuvres_export = df_oeuvres.copy()
df_planning_export = df_planning.copy()

for df in [df_catalogue_export, df_oeuvres_export, df_planning_export]:
    for colonne in df.columns:
        if df[colonne].apply(lambda value: isinstance(value, (list, dict))).any():
            df[colonne] = df[colonne].apply(lambda value: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else value)

df_catalogue_export.to_csv(fichier_catalogue_csv, index=False, encoding="utf-8-sig")
df_catalogue.to_json(fichier_catalogue_json, orient="records", force_ascii=False, indent=2)

df_oeuvres_export.to_csv(fichier_oeuvres_csv, index=False, encoding="utf-8-sig")
df_oeuvres.to_json(fichier_oeuvres_json, orient="records", force_ascii=False, indent=2)

df_planning_export.to_csv(fichier_planning_csv, index=False, encoding="utf-8-sig")
df_planning.to_json(fichier_planning_json, orient="records", force_ascii=False, indent=2)

print("Exports créés :")
for fichier in [
    fichier_catalogue_csv,
    fichier_catalogue_json,
    fichier_oeuvres_csv,
    fichier_oeuvres_json,
    fichier_planning_csv,
    fichier_planning_json,
]:
    print(fichier.resolve())

Exports créés :
C:\TCHIKSON\Webscrapping\exports\anime_sama_catalogue.csv
C:\TCHIKSON\Webscrapping\exports\anime_sama_catalogue.json
C:\TCHIKSON\Webscrapping\exports\anime_sama_oeuvres_enrichies.csv
C:\TCHIKSON\Webscrapping\exports\anime_sama_oeuvres_enrichies.json
C:\TCHIKSON\Webscrapping\exports\anime_sama_planning.csv
C:\TCHIKSON\Webscrapping\exports\anime_sama_planning.json


In [42]:
%pip install yt-dlp 
%pip install imageio-ffmpeg


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Étape 11 - fonction de Téléchargement


In [64]:
import json
import unicodedata
import difflib
import urllib.parse
import os
import tempfile
import subprocess
import shutil
import yt_dlp


# ============ CONFIG ============
PRIORITE_HOSTS = [
    "sibnet", "sendvid", "dingtezuni", "minochinos",
    "vidmoly", "embed4me", "vk.com", "anime-sama.fr",
]


# ============ RECHERCHE ============
def charger_catalogue(chemin_json):
    with open(chemin_json, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        for cle in ("animes", "fiches", "data", "results"):
            if cle in data and isinstance(data[cle], list):
                return data[cle]
        return [data]
    return data


def _normaliser(texte):
    if not texte:
        return ""
    texte = unicodedata.normalize("NFKD", texte)
    texte = "".join(c for c in texte if not unicodedata.combining(c))
    return "".join(c.lower() for c in texte if c.isalnum() or c.isspace()).strip()


def rechercher_oeuvre(catalogue, *, rank=None, title=None, seuil=0.6):
    if rank is not None:
        for oeuvre in catalogue:
            if oeuvre.get("rank") == rank:
                return oeuvre
        return None
    if title is not None:
        cible = _normaliser(title)
        for oeuvre in catalogue:
            titres = [oeuvre.get("title", "")] + oeuvre.get("alternative_titles", [])
            if any(_normaliser(t) == cible for t in titres):
                return oeuvre
        for oeuvre in catalogue:
            titres = [oeuvre.get("title", "")] + oeuvre.get("alternative_titles", [])
            if any(cible in _normaliser(t) for t in titres if t):
                return oeuvre
        meilleur, meilleur_score = None, 0.0
        for oeuvre in catalogue:
            titres = [oeuvre.get("title", "")] + oeuvre.get("alternative_titles", [])
            for t in titres:
                score = difflib.SequenceMatcher(None, cible, _normaliser(t)).ratio()
                if score > meilleur_score:
                    meilleur, meilleur_score = oeuvre, score
        if meilleur_score >= seuil:
            return meilleur
    return None
# --- FIX 1 : localiser ffmpeg (PATH système, sinon imageio-ffmpeg) ---
def _trouver_ffmpeg():
    chemin = shutil.which("ffmpeg")
    if chemin:
        return chemin
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return None

FFMPEG = _trouver_ffmpeg()
if FFMPEG is None:
    raise RuntimeError("ffmpeg introuvable. Fais : pip install imageio-ffmpeg")
print("ffmpeg utilisé :", FFMPEG)

def _referer_pour(url: str) -> str:
    """Renvoie le Referer attendu par le host (souvent l'origine du domaine)."""
    p = urllib.parse.urlparse(url)
    return f"{p.scheme}://{p.netloc}/"

def extract_stream_urls(embed_url: str) -> list[str]:
    """Retourne tous les liens .m3u8 ou .mp4 trouvés depuis un embed."""

    ydl_opts = {
        "quiet": True,
        "skip_download": True,
        "no_warnings": True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(embed_url, download=False)

    urls = []

    for f in info.get("formats", []):
        url = f.get("url", "")

        if url and (".m3u8" in url or ".mp4" in url):
            urls.append(url)

    if info.get("url"):
        urls.append(info["url"])

    # Retire les doublons en conservant l'ordre
    urls = list(dict.fromkeys(urls))

    if not urls:
        raise RuntimeError("Aucun flux vidéo trouvé")

    return urls


def download_to_tempfile(stream_url: str, filename: str, referer: str = None, timeout: int = 300) -> str:
    """Télécharge la vidéo dans un dossier temporaire et renvoie le chemin du MP4."""
    temp_dir = tempfile.mkdtemp(prefix="video_dl_")
    output_path = os.path.join(temp_dir, filename)
    # --- FIX 2 : on utilise FFMPEG (chemin complet) au lieu de "ffmpeg" ---
    cmd = [FFMPEG, "-y"]
    # Headers AVANT -i : sinon ignorés
    if referer:
        cmd += ["-headers", f"Referer: {referer}\r\n"]
    cmd += [
        "-user_agent", "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "-rw_timeout", "15000000",   # 15s sans données -> abandon (microsecondes)
        "-i", stream_url,
        "-c", "copy",
        "-bsf:a", "aac_adtstoasc",
        output_path,
    ]
    print("Téléchargement dans :", output_path)
    subprocess.run(cmd, check=True, timeout=timeout)
    return output_path


def download_episode_temp(embed_url: str, title: str) -> str:
    """Pipeline pour un lecteur : extrait le flux puis télécharge."""

    print(f"\n=== Téléchargement : {title} ===")
    print("1. URL du lecteur :", repr(embed_url))

    try:
        stream = extract_stream_url(embed_url)
    except Exception as e:
        print("Échec pendant extract_stream_url()")
        print("URL du lecteur :", repr(embed_url))
        print("Type :", type(e).__name__)
        print("Détail :", e)
        raise

    print("2. Flux extrait :", repr(stream))

    if not stream:
        raise RuntimeError(
            f"Aucun flux extrait depuis le lecteur {embed_url!r}"
        )

    safe_title = (
        title
        .replace(" ", "_")
        .replace("/", "-")
        .replace('"', "")
        .replace("'", "")
        + ".mp4"
    )

    try:
        referer = _referer_pour(stream)
    except Exception as e:
        print("Échec pendant _referer_pour()")
        print("Flux :", repr(stream))
        print("Type :", type(e).__name__)
        print("Détail :", e)
        raise

    print("3. Referer :", repr(referer))

    try:
        temp_file = download_to_tempfile(
            stream,
            safe_title,
            referer=referer
        )
    except Exception as e:
        print("Échec pendant download_to_tempfile()")
        print("Flux :", repr(stream))
        print("Referer :", repr(referer))
        print("Type :", type(e).__name__)
        print("Détail :", e)
        raise

    print("Fichier temporaire :", temp_file)
    return temp_file


def recuperer_lecteurs_ordonnes(oeuvre, *, saison=1, episode=1):
    """Liste des URLs de lecteurs pour (saison, episode), triée par priorité de host."""
    saisons = oeuvre.get("episodes_by_season", [])
    if not saisons:
        return []

    bloc = None
    if isinstance(saison, int):
        if 1 <= saison <= len(saisons):
            bloc = saisons[saison - 1]
    else:
        for s in saisons:
            if _normaliser(s.get("season_label", "")) == _normaliser(saison):
                bloc = s
                break
    if bloc is None:
        return []

    def rang(url):
        for i, host in enumerate(PRIORITE_HOSTS):
            if host in url:
                return i
        return len(PRIORITE_HOSTS)

    for ep in bloc.get("iframes", []):
        if ep.get("episode") == episode:
            return sorted(ep.get("lecteurs", []), key=rang)
    return []


def download_episode_avec_fallback(oeuvre, *, saison=1, episode=1):
    """Essaie chaque lecteur jusqu'au premier téléchargement réussi."""
    lecteurs = recuperer_lecteurs_ordonnes(oeuvre, saison=saison, episode=episode)
    if not lecteurs:
        raise RuntimeError(f"Aucun lecteur pour S{saison}E{episode}")

    titre = f"{oeuvre['title']}_S{saison}E{episode}"
    derniere_erreur = None
    for url in lecteurs:
        try:
            print(f"-> tentative : {url}")
            return download_episode_temp(url, titre)
        # --- FIX 3 : ffmpeg manquant -> inutile d'essayer les autres lecteurs ---
        except FileNotFoundError as e:
            raise RuntimeError(
                "ffmpeg introuvable -> pip install imageio-ffmpeg (puis relance la cellule)"
            ) from e
        except subprocess.TimeoutExpired as e:
            derniere_erreur = e
            print(f"   timeout (s dépassés), lecteur suivant…")
        except Exception as e:
            derniere_erreur = e
            print(f"   échec ({type(e).__name__}: {e}), lecteur suivant…")
    raise RuntimeError(f"Tous les lecteurs ont échoué pour {titre}") from derniere_erreur

ffmpeg utilisé : c:\Users\33610\AppData\Local\Programs\Python\Python313\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe


## Utilisation — recherche puis récupération de la vidéo

In [69]:

import difflib
catalogue = charger_catalogue("animeSama/exports/anime_sama_oeuvres_enrichies.json")

oeuvre = rechercher_oeuvre(catalogue, title="100 Meters")
if oeuvre is None:
    print("Aucune œuvre trouvée")
else:
    print("Trouvé :", oeuvre.get("title"))
    chemin = download_episode_avec_fallback(oeuvre, saison="Film", episode=1)
    print("Téléchargé :", chemin)

Trouvé : 100 Meters
-> tentative : https://vidmoly.net/embed-lxwtjlaimdac.html

=== Téléchargement : 100 Meters_SFilmE1 ===
1. URL du lecteur : 'https://vidmoly.net/embed-lxwtjlaimdac.html'
2. Flux extrait : 'https://prx-1329-ant-t.vmwesa.online/hls2/01/02361/i9rdljqyv3e7_l/index-v1-a1.m3u8?t=NJnwBln7PMmvy2nN_jPgBY2wBHn0pr6ekwxFDihSZIg=&s=1780664290&e=43200&v=&srv=bck-1331-ant&i=0.4&sp=0&asn=199636'
3. Referer : 'https://prx-1329-ant-t.vmwesa.online/'
Téléchargement dans : C:\Users\33610\AppData\Local\Temp\video_dl_5tt46dnl\100_Meters_SFilmE1.mp4
Échec pendant download_to_tempfile()
Flux : 'https://prx-1329-ant-t.vmwesa.online/hls2/01/02361/i9rdljqyv3e7_l/index-v1-a1.m3u8?t=NJnwBln7PMmvy2nN_jPgBY2wBHn0pr6ekwxFDihSZIg=&s=1780664290&e=43200&v=&srv=bck-1331-ant&i=0.4&sp=0&asn=199636'
Referer : 'https://prx-1329-ant-t.vmwesa.online/'
Type : CalledProcessError
Détail : Command '['c:\\Users\\33610\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\imageio_ffmpeg\\binaries\\ffm

RuntimeError: Tous les lecteurs ont échoué pour 100 Meters_SFilmE1